# 🎯 Tutorial 07: Skill Discovery y Composición

## Aprender Habilidades Reutilizables

En este tutorial aprenderás:

- 🧩 Qué son las "skills" o primitivas de comportamiento
- 🔨 Cómo descubrir habilidades reutilizables automáticamente
- 🎼 Composición de habilidades para resolver tareas complejas
- 💻 Implementación básica de skill discovery

---

## 📖 Parte 1: Teoría - Habilidades como Building Blocks

### El Problema:

Los humanos aprenden **habilidades reutilizables** que pueden combinar:
- 🏃 Caminar
- 🤲 Agarrar
- 🔄 Girar
- 📍 Navegar

Para resolver tareas complejas: "Trae el vaso de la cocina"
= Navegar(cocina) + Agarrar(vaso) + Navegar(aquí)

### Skill Discovery:

En lugar de aprender cada tarea desde cero:
1. **Descubre** un conjunto de skills básicas
2. **Aprende** cuándo usar cada skill
3. **Compone** skills para nuevas tareas

### Métodos Principales:

#### 1. **Diversity-Driven Discovery**
- Aprende skills que llevan a estados diversos
- Maximiza mutual information: $I(s;z)$ donde $z$ es el skill

#### 2. **Successor Features**
- Representa skills por su "firma" de features
- Permite transfer entre tareas relacionadas

#### 3. **Option Discovery**
- Options: políticas temporalmente extendidas
- Aprende cuándo iniciar/terminar cada option

### Ventajas:

- ✅ **Transfer eficiente**: Skills aprendidas en una tarea ayudan en otras
- ✅ **Exploración estructurada**: Skills guían la exploración
- ✅ **Interpretabilidad**: Skills son comprensibles
- ✅ **Composición**: Resolver tareas complejas componiendo skills simples


---

## 🛠️ Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from utils.test_utils import print_success
from utils.data_utils import set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ Setup completo!")

---

## 💻 Parte 3: Ejemplo Conceptual - Grid World

Vamos a ver un ejemplo simplificado de skill discovery en un grid world.

In [ ]:
class SimpleGridWorld:
    """
    Grid world simple para demostrar skill discovery.
    
    El agente puede aprender skills como:
    - "Ir a esquina superior izquierda"
    - "Ir a centro"
    - "Explorar bordes"
    """
    
    def __init__(self, size=5):
        self.size = size
        self.reset()
    
    def reset(self):
        self.pos = [self.size // 2, self.size // 2]  # Centro
        return self.get_state()
    
    def get_state(self):
        state = np.zeros((self.size, self.size))
        state[self.pos[0], self.pos[1]] = 1
        return state.flatten()
    
    def step(self, action):
        # Actions: 0=up, 1=right, 2=down, 3=left
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        move = moves[action]
        
        new_pos = [
            max(0, min(self.size - 1, self.pos[0] + move[0])),
            max(0, min(self.size - 1, self.pos[1] + move[1]))
        ]
        
        self.pos = new_pos
        return self.get_state(), 0, False  # state, reward, done


class SkillPolicy(nn.Module):
    """
    Política condicionada en un skill z.
    
    π(a | s, z): dado estado s y skill z, elige acción a
    """
    
    def __init__(self, state_dim, n_skills=4, n_actions=4, hidden=64):
        super().__init__()
        self.n_skills = n_skills
        
        # Input: estado + one-hot skill
        self.fc1 = nn.Linear(state_dim + n_skills, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_actions)
    
    def forward(self, state, skill):
        """ 
        Args:
            state: [batch, state_dim]
            skill: [batch] - índice del skill
        
        Returns:
            action_probs: [batch, n_actions]
        """
        skill_onehot = F.one_hot(skill, self.n_skills).float()
        x = torch.cat([state, skill_onehot], dim=-1)
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        action_logits = self.fc3(x)
        
        return F.softmax(action_logits, dim=-1)


print("✅ Entorno y políticas de skills definidas!")

---

## 📊 Parte 4: Visualización Conceptual

Veamos cómo diferentes skills pueden llevar a comportamientos distintos.

In [ ]:
# Crear entorno y política
env = SimpleGridWorld(size=5)
policy = SkillPolicy(state_dim=25, n_skills=4, n_actions=4)

# Simular diferentes skills
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for skill_idx in range(4):
    ax = axes[skill_idx // 2, skill_idx % 2]
    
    # Rollout con este skill
    state = env.reset()
    trajectory = [env.pos.copy()]
    
    for _ in range(10):
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        skill_tensor = torch.LongTensor([skill_idx])
        
        with torch.no_grad():
            action_probs = policy(state_tensor, skill_tensor)
            action = torch.multinomial(action_probs, 1).item()
        
        state, _, _ = env.step(action)
        trajectory.append(env.pos.copy())
    
    # Visualizar
    grid = np.zeros((5, 5))
    for i, pos in enumerate(trajectory):
        grid[pos[0], pos[1]] = (i + 1) / len(trajectory)
    
    im = ax.imshow(grid, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Skill {skill_idx}', fontsize=14, fontweight='bold')
    ax.axis('off')
    
    # Plot trajectory
    for i in range(len(trajectory) - 1):
        ax.arrow(trajectory[i][1], trajectory[i][0],
                trajectory[i+1][1] - trajectory[i][1],
                trajectory[i+1][0] - trajectory[i][0],
                color='red', width=0.05, head_width=0.2, alpha=0.6)

plt.suptitle('Diferentes Skills = Diferentes Comportamientos', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📍 Con entrenamiento apropiado, cada skill aprendería un comportamiento útil")
print("   Ejemplo: Skill 0 = 'ir a esquina', Skill 1 = 'ir al centro', etc.")

---

## 🎓 Resumen y Aplicaciones

### ✅ Lo que aprendiste:

1. **Skills** son comportamientos reutilizables y componibles
2. **Skill Discovery** automatiza el aprendizaje de estas primitivas
3. Los skills permiten **transfer** eficiente entre tareas
4. La **composición** de skills resuelve problemas complejos

### 🌟 Aplicaciones Reales:

#### 🤖 Robótica:
- **Manipulación**: Skills = {agarrar, soltar, empujar, girar}
- **Locomoción**: Skills = {caminar, correr, subir escaleras, saltar}
- **Navegación**: Skills = {ir_a_punto, seguir_pared, evitar_obstáculo}

#### 🎮 Juegos:
- **Minecraft**: Skills = {minar, construir, buscar_recursos}
- **StarCraft**: Skills = {rush, defend, macro, scout}

#### 🏭 Automatización:
- **Manufactura**: Skills = {soldar, ensamblar, inspeccionar}
- **Logística**: Skills = {recoger, clasificar, empaquetar, transportar}

### 📚 Papers Clave:

- **DIAYN**: [Diversity is All You Need](https://arxiv.org/abs/1802.06070)
- **Option-Critic**: [The Option-Critic Architecture](https://arxiv.org/abs/1609.05140)
- **Successor Features**: [Universal Value Function Approximators](https://arxiv.org/abs/1506.05254)

### 🔮 Futuro:

La combinación de **Meta-Learning + Skill Discovery** es prometedora:
- Aprender a descubrir skills rápidamente en nuevos dominios
- Transfer de skills entre robots diferentes
- Lifelong learning acumulando biblioteca de skills

---

## 🎉 ¡Has completado el tutorial de Skill Discovery!

Este es un área activa de investigación con mucho potencial para IA verdaderamente adaptativa.
